In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

import numpy as np
import matplotlib

import matplotlib.pyplot as plt

import time
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans
import umap
from sklearn.cluster import DBSCAN
from sklearn import metrics

from tqdm import tqdm_notebook
from lmfit import minimize, Parameters
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as pl
import shap

import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
%load_ext autoreload
%autoreload 2

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/BRCA_SHAP/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
from PermCell_Smooth import *
#from SHAPset import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
Run="exp03"
import xgboost as xgb
import umap
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import pandas as pd

from cytof_transform import *

# Load and initialize

In [ ]:
import glob
from borb.pdf import Document, Page, PageLayout, SingleColumnLayout, Paragraph, PDF


In [ ]:
dir="/Users/ronguy/Dropbox/CyTOF_Breast/CyTOF_CR7/CyTOF7_scMCF7/csv_scale_value/"

In [ ]:
FList=glob.glob(dir+"*")

In [ ]:
sc.set_figure_params()

In [ ]:
FList.sort()
FList

In [ ]:
DBs=[f"exp{i:02}" for i in range(1,7)]

In [ ]:
for F,DB in zip(FList,DBs):
    print(F)
    globals()[DB]=pd.read_csv(F,)

In [ ]:
Rep=dict(zip(list(exp01.columns),[f.split("_")[-1] for f in list(exp01.columns)]))

In [ ]:
Rep

In [ ]:
for DB in DBs:
    globals()[DB].rename(columns=Rep,inplace=True)

In [ ]:
#dir="/Users/ronguy/Dropbox/WIS-CIMA colab - Analysis/#3 CyTOF  - KPC sample, after CD45 depletion/"

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
Rep=dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:,:].values)

In [ ]:
Rep['H3K27Ac']='H3K27ac'

In [ ]:
for F,DB in zip(FList,DBs):
#    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)
#    globals()[DB].drop(columns=['DNA1','DNA2','Event #'],inplace=True)

In [ ]:
N=list(exp01.columns)
N.sort()


In [ ]:
N

In [ ]:
DBs=[Run]

In [ ]:
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)

In [ ]:
EpiCols.remove('BMI1')
CellIden.append('BMI1')

In [ ]:
EpiCols.remove('EZH2')
CellIden.append('EZH2')

In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
len(NamesAll)

In [ ]:
for DB in DBs:
    plt.figure()
    D=np.arcsinh(globals()[DB]/5).copy()
    sns.histplot(data=D,x='H3',**hKWD,color='blue')
    sns.histplot(data=D,x='H3.3',**hKWD,color='red')
#    sns.histplot(data=D,x='KRT8-18',**hKWD,color='magenta')
    sns.histplot(data=D,x='H4',**hKWD,color='g')
    plt.title(DB)
#plt.xscale('log')
#plt.yscale('log')

## Gate on H3.3/H2A too low, but also remove outliers 99.99% from all 

In [ ]:
GateColumns=['H3.3','H4','H3']



def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
#    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data




In [ ]:
for DB in DBs:
    globals()[DB]=Gate(globals()[DB],DB)



In [ ]:
scFac=5
for DB in DBs:
    globals()[DB]=np.arcsinh(globals()[DB]/scFac)


In [ ]:
control_markers = ["H3.3", "H3", "H4", ]
markers_to_correct = NormMRK

In [ ]:
compute_marker_tech_correlations(exp03,None,control_markers);

In [ ]:
globals()[f"{DBs[0]}_UnCorr"]=globals()[f"{DBs[0]}"].copy()

In [ ]:
cfg = CytofTransformConfig(
    control_markers=control_markers,
    markers_to_correct=markers_to_correct,
    use_compartments=False,
    n_pcs_for_T=1,
    anchor_to_median=True,
    zscore=False,
    line_col=None
)
asinh_data = globals()[DBs[0]].copy()

result_global = cytof_transform_global(asinh_data, cfg)
asinh_corr_global = result_global.corrected.copy()
zres_global = result_global.residuals_z.copy()

In [ ]:
globals()[DBs[0]]=asinh_corr_global.copy()

In [ ]:
sns.set_theme(palette="bright")
DefStyle()


In [ ]:
globals()[DBs[0]]=(globals()[DBs[0]]-globals()[DBs[0]].mean(axis=0))/globals()[DBs[0]].std(axis=0)

In [ ]:
tech1, loadings, ev = plot_tech_factor_qc(
    asinh_data=globals()[f"{DBs[0]}_UnCorr"],
    control_markers=["H3.3", "H3", "H4"],
    tech_factor=None,    # or result_global.tech_factor
    tech_name="tech1",
)

In [ ]:
corr_df = plot_marker_correlations_qc(
    asinh_pre=globals()[f"{DBs[0]}_UnCorr"],               # unnormalized
    asinh_post=globals()[f"{DBs[0]}"],  # corrected
    tech_factor=result_global.tech_factor,
    # markers_to_highlight=[
    #     "H3.3", "H3", "H4",
    #     "H3K27ac", "H3K4me3", "H3K9ac",
    #     "KI67"
    # ] ,
    top_n=35,
)

In [ ]:
sns.set_palette("bright")

In [ ]:
marker_groups = {
    "core_histones": ["H3.3", "H3", "H4"],
    "PTMs": ["H3K27ac", "H3K4me3", "H3K9ac", "H3K27me3", "H3K36me3"],
    "proliferation": ["KI67"],
    "epithelial": ["ER", "GATA3", "KRT8-18", "KRT5"],
}

plot_gamma_qc(result_global.gamma, marker_groups=marker_groups)

In [ ]:
import numpy as np
import pandas as pd

def er_high_low_labels(asinh_data: pd.DataFrame,
                       er_marker: str = "ER",
                       low_q: float = 0.2,
                       high_q: float = 0.8) -> pd.Series:
    """
    Label cells as 'ER_low', 'ER_high', or NaN (middle) based on quantiles.

    Parameters
    ----------
    asinh_data : DataFrame
        Cells × markers (arcsinh).
    er_marker : str
        Column name of the ER marker.
    low_q, high_q : float
        Quantiles defining low and high ER.

    Returns
    -------
    labels : Series
        Index = cells, values = 'ER_low', 'ER_high', or NaN.
    """
    er_vals = asinh_data[er_marker]
    q_low = er_vals.quantile(low_q)
    q_high = er_vals.quantile(high_q)

    labels = pd.Series(index=asinh_data.index, dtype="object")
    labels[er_vals <= q_low] = "ER_low"
    labels[er_vals >= q_high] = "ER_high"
    # middle cells left as NaN
    return labels


def bio_group_diff_pre_post(
    asinh_pre: pd.DataFrame,
    asinh_post: pd.DataFrame,
    group_labels: pd.Series,
    group1: str,
    group2: str,
    markers: list[str] | None = None,
) -> pd.DataFrame:
    """
    Compare marker means for two biological groups, pre vs post normalization.

    Parameters
    ----------
    asinh_pre, asinh_post : DataFrame
        Cells × markers (same index, same columns).
    group_labels : Series
        Index must match data; values are group names (e.g. 'ER_low', 'ER_high').
    group1, group2 : str
        Two group labels to compare (group1 - group2).
    markers : list of str, optional
        Subset of markers to compare. If None, use all columns.

    Returns
    -------
    df : DataFrame
        Columns:
          mean1_pre, mean2_pre, diff_pre,
          mean1_post, mean2_post, diff_post
        Rows = markers.
    """
    assert asinh_pre.index.equals(asinh_post.index)
    assert asinh_pre.index.equals(group_labels.index)

    if markers is None:
        markers = list(asinh_pre.columns)

    mask1 = group_labels == group1
    mask2 = group_labels == group2

    pre1  = asinh_pre.loc[mask1, markers].mean()
    pre2  = asinh_pre.loc[mask2, markers].mean()
    post1 = asinh_post.loc[mask1, markers].mean()
    post2 = asinh_post.loc[mask2, markers].mean()

    df = pd.DataFrame({
        "mean1_pre": pre1,
        "mean2_pre": pre2,
        "diff_pre": pre1 - pre2,
        "mean1_post": post1,
        "mean2_post": post2,
        "diff_post": post1 - post2,
    })
    return df


In [ ]:
import numpy as np
import pandas as pd


def compute_effect_sizes_er(
    asinh_pre: pd.DataFrame,
    asinh_post: pd.DataFrame,
    er_labels: pd.Series,
    markers: list[str],
    group1: str = "ER_high",
    group2: str = "ER_low",
) -> pd.DataFrame:
    """
    Compute Cohen's d effect sizes for ER_high vs ER_low, pre and post normalization.

    Parameters
    ----------
    asinh_pre, asinh_post : DataFrame
        Cells × markers (same index, same columns).
    er_labels : Series
        Index must match data; values should include group1, group2, and possibly NaN for middle.
    markers : list of str
        Markers to compute effect sizes for.
    group1, group2 : str
        Names of the two groups (default: 'ER_high' vs 'ER_low').

    Returns
    -------
    df : DataFrame
        Index = markers
        Columns:
            n1, n2
            mean1_pre, mean2_pre, diff_pre, sd_pooled_pre, d_pre
            mean1_post, mean2_post, diff_post, sd_pooled_post, d_post
            ratio_diff_post_to_pre
    """
    assert asinh_pre.index.equals(asinh_post.index)
    assert asinh_pre.index.equals(er_labels.index)

    mask1 = er_labels == group1
    mask2 = er_labels == group2

    if mask1.sum() == 0 or mask2.sum() == 0:
        raise ValueError("One of the groups has zero cells.")

    records = []

    for m in markers:
        if m not in asinh_pre.columns:
            continue

        x1_pre = asinh_pre.loc[mask1, m].values
        x2_pre = asinh_pre.loc[mask2, m].values
        x1_post = asinh_post.loc[mask1, m].values
        x2_post = asinh_post.loc[mask2, m].values

        n1 = len(x1_pre)
        n2 = len(x2_pre)

        # Pre
        mean1_pre = np.mean(x1_pre)
        mean2_pre = np.mean(x2_pre)
        diff_pre = mean1_pre - mean2_pre
        sd1_pre = np.std(x1_pre, ddof=1)
        sd2_pre = np.std(x2_pre, ddof=1)
        # pooled SD (equal variances)
        sd_pooled_pre = np.sqrt(
            ((n1 - 1) * sd1_pre**2 + (n2 - 1) * sd2_pre**2) / (n1 + n2 - 2)
        )
        d_pre = diff_pre / sd_pooled_pre if sd_pooled_pre > 0 else np.nan

        # Post
        mean1_post = np.mean(x1_post)
        mean2_post = np.mean(x2_post)
        diff_post = mean1_post - mean2_post
        sd1_post = np.std(x1_post, ddof=1)
        sd2_post = np.std(x2_post, ddof=1)
        sd_pooled_post = np.sqrt(
            ((n1 - 1) * sd1_post**2 + (n2 - 1) * sd2_post**2) / (n1 + n2 - 2)
        )
        d_post = diff_post / sd_pooled_post if sd_pooled_post > 0 else np.nan

        ratio = diff_post / diff_pre if diff_pre != 0 else np.nan

        records.append(
            {
                "marker": m,
                "n1": n1,
                "n2": n2,
                "mean1_pre": mean1_pre,
                "mean2_pre": mean2_pre,
                "diff_pre": diff_pre,
                "sd_pooled_pre": sd_pooled_pre,
                "d_pre": d_pre,
                "mean1_post": mean1_post,
                "mean2_post": mean2_post,
                "diff_post": diff_post,
                "sd_pooled_post": sd_pooled_post,
                "d_post": d_post,
                "ratio_diff_post_to_pre": ratio,
            }
        )

    df = pd.DataFrame.from_records(records).set_index("marker")
    return df


In [ ]:
# 1) label ER-high / ER-low using pre-normalization values
er_labels = er_high_low_labels(globals()[f"{DBs[0]}_UnCorr"], er_marker="ER", low_q=0.2, high_q=0.8)

# 2) choose markers to inspect (add whatever you care about)
markers_to_check = [
    "ER", "GATA3", "KRT8-18", "KRT5", "Vimentin",
    "H3.3", "H3", "H4", "H3K27ac", "H3K4me3", "KI67"
]

# 3) compute x̄_high - x̄_low pre vs post
er_diff_df = bio_group_diff_pre_post(
    asinh_pre=globals()[f"{DBs[0]}_UnCorr"],
    asinh_post=globals()[f"{DBs[0]}"],
    group_labels=er_labels,
    group1="ER_high",
    group2="ER_low",
    markers=markers_to_check,
)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


def plot_er_violin_pre_post(
    asinh_pre: pd.DataFrame,
    asinh_post: pd.DataFrame,
    er_labels: pd.Series,
    markers_to_plot: list[str],
    group1: str = "ER_high",
    group2: str = "ER_low",
    figsize_per_marker: tuple = (6, 4),
):
    """
    Violin plots for ER_high vs ER_low, before vs after normalization.

    Parameters
    ----------
    asinh_pre, asinh_post : DataFrame
        Cells × markers.
    er_labels : Series
        'ER_high' / 'ER_low' (other/middle values will be dropped).
    markers_to_plot : list of str
        Markers to show.
    group1, group2 : str
        Names of the two ER groups.
    figsize_per_marker : tuple
        Size of each marker row (width, height). Total fig size scales with len(markers).
    """
    assert asinh_pre.index.equals(asinh_post.index)
    assert asinh_pre.index.equals(er_labels.index)

    # Only use the two ER groups; drop middle
    mask = er_labels.isin([group1, group2])
    labels = er_labels[mask]

    # Build long-format dataframe
    records = []
    for state_name, df_state in [("Before", asinh_pre), ("After", asinh_post)]:
        df_state = df_state.loc[mask, :]
        for m in markers_to_plot:
            if m not in df_state.columns:
                continue
            vals = df_state[m].values
            for v, g in zip(vals, labels.values):
                records.append(
                    {
                        "marker": m,
                        "value": v,
                        "ER_group": g,
                        "state": state_name,
                    }
                )

    long_df = pd.DataFrame.from_records(records)

    # Determine figure size: rows = markers, 2 cols (Before/After)
    n_markers = len(markers_to_plot)
    fig_width = figsize_per_marker[0] * 2
    fig_height = figsize_per_marker[1] * n_markers

    sns.set(style="whitegrid")
    fig, axes = plt.subplots(
        nrows=n_markers,
        ncols=2,
        figsize=(fig_width, fig_height),
        squeeze=False,
    )

    for i, m in enumerate(markers_to_plot):
        sub = long_df[long_df["marker"] == m]

        for j, state_name in enumerate(["Before", "After"]):
            ax = axes[i, j]
            sub_state = sub[sub["state"] == state_name]

            sns.violinplot(
                data=sub_state,
                x="ER_group",
                y="value",
                ax=ax,
                cut=0,
                inner="box",
            )
            ax.set_title(f"{m} – {state_name}")
            ax.set_xlabel("")
            if j == 0:
                ax.set_ylabel("asinh intensity")
            else:
                ax.set_ylabel("")

    # Label bottom row x-axis
    for j in range(2):
        axes[-1, j].set_xlabel("ER group")

    plt.tight_layout()
    plt.show()


In [ ]:
result_global.corrected